# Lab 3: Quadratic Knapsack Problem and PUBO, Classical Methods and QUBO

**Course:** Quantum Optimization Lab 3

**Topics covered:** Quadratic Knapsack Problem (QKP), Polynomial Unconstrained Binary Optimization (PUBO), Rosenberg quadratization

**Tools:** dimod, dwave-neal, numpy, matplotlib

## Learning objectives

By the end of this lab you should be able to:

1. State the Quadratic Knapsack Problem and the general PUBO problem, and explain how each generalizes the plain 0-1 knapsack from Lecture 7.
2. Solve small instances of both problems by brute force and by a greedy heuristic, and explain in your own words why the greedy heuristic can miss the synergy bonuses that make these problems interesting in the first place.
3. Build the QUBO matrix $Q$ for a Quadratic Knapsack instance directly from the objective and the capacity penalty, the same way it was derived in Lecture 7.
4. Apply the Rosenberg quadratization to reduce a cubic PUBO term to a quadratic one, and verify by hand that the auxiliary penalty enforces the correct substitution.
5. Solve both QUBOs with simulated annealing and compare the result against brute force and greedy on value, feasibility, and runtime.

## How this notebook is organized

- **Part 1** works through the Quadratic Knapsack Problem: theory, worked example, brute force, greedy, QUBO formulation, and simulated annealing.
- **Part 2** works through PUBO: theory, a worked cubic-term example, brute force, greedy local search, Rosenberg quadratization down to a QUBO, and simulated annealing.
- **Part 3** is a set of exercises to complete on your own.

Run every code cell in order. The **practice task** call-outs are meant to be done for your practice; small changes to a parameter and a re-run, not new code.

In [ ]:
!pip install dimod dwave-neal -q

In [ ]:
import itertools
import time
import numpy as np
import matplotlib.pyplot as plt
import dimod
import neal

RNG_SEED = 11
rng = np.random.default_rng(RNG_SEED)
sampler = neal.SimulatedAnnealingSampler()

print("dimod version:", dimod.__version__)
print("neal version:", neal.__version__)

---
# Part 1: Quadratic Knapsack Problem (QKP)

The 0-1 knapsack from Lecture 7 assumes every item contributes its value independently: $\sum_j v_j x_j$. In practice, choosing two items *together* is sometimes worth more, or less, than the sum of their individual values. The Quadratic Knapsack Problem (QKP) adds a pairwise synergy term $q_{ij}$ for every pair of items:

$$\max \sum_{j=1}^{n} v_j x_j + \sum_{i<j} q_{ij} x_i x_j \qquad \text{subject to} \qquad \sum_{j=1}^{n} w_j x_j \le W, \quad x_j \in \{0,1\}$$

Here $q_{ij}$ is the extra value gained (or lost, if negative) when items $i$ and $j$ are both packed. Setting every $q_{ij} = 0$ recovers the plain 0-1 knapsack exactly.

## 1.1 Why anyone outside a classroom cares about this

- **Project portfolio selection.** Two R&D projects that share infrastructure or a research team can be worth more together than the sum of their individual returns.
- **Asset allocation.** Correlated or complementary assets in a portfolio can add diversification value beyond each asset's own expected return.
- **Protein and drug design.** Pairwise amino acid interaction energies determine whether a set of residues folds into a stable structure, which is naturally a quadratic objective over binary choices.
- **Media and advertising mix.** Two channels shown together can reinforce each other's effect on a customer (or cannibalize each other), which shows up as a pairwise synergy term in the campaign budget allocation problem.

## 1.2 A worked example you can check by hand

Four projects, weight budget $W = 16$:

| Item $j$ | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| Value $v_j$ | 2 | 5 | 2 | 4 |
| Weight $w_j$ | 8 | 6 | 5 | 3 |

Pairwise synergies: $q_{12}=8,\ q_{13}=6,\ q_{14}=10,\ q_{23}=2,\ q_{24}=6,\ q_{34}=4$.

Try packing items 1, 3, and 4: weight $= 8+5+3=16 \le 16$, and the value is $2+2+4 + q_{13}+q_{14}+q_{34} = 8 + 6+10+4 = 28$. Keep this number, we will recover it from brute force, from the QUBO, and from simulated annealing below.

In [ ]:
# The worked example above, kept as plain Python so every method in this
# section can be run against the same instance and checked by hand.
example_values = [2, 5, 2, 4]
example_weights = [8, 6, 5, 3]
example_capacity = 16
# synergy stored as {(i, j): q_ij} with 0-indexed items and i < j
example_synergy = {
    (0, 1): 8,
    (0, 2): 6,
    (0, 3): 10,
    (1, 2): 2,
    (1, 3): 6,
    (2, 3): 4,
}

def qkp_value(values, synergy, x):
    '''Objective value of a QKP solution x (a 0/1 sequence), ignoring feasibility.'''
    total = sum(v * xi for v, xi in zip(values, x))
    for (i, j), q in synergy.items():
        total += q * x[i] * x[j]
    return total

def qkp_weight(weights, x):
    return sum(w * xi for w, xi in zip(weights, x))

# sanity check against the worked example above
x_check = (1, 0, 1, 1)
print("weight of (1,0,1,1):", qkp_weight(example_weights, x_check))
print("value of (1,0,1,1): ", qkp_value(example_values, example_synergy, x_check))

## 1.3 Classical approaches

We look at the same two classical strategies as in previous labs: brute force, which is exact but exponential, and a greedy heuristic, which is fast but has no way of seeing the pairwise synergies coming.

### 1.3.1 Brute force

For $n$ items, brute force checks every one of the $2^n$ subsets, discards the ones that exceed capacity, and keeps the feasible subset with the highest value (linear value plus every pairwise synergy inside the subset). This is exact and, exactly like plain knapsack, useless past a few dozen items.

In [ ]:
def qkp_brute_force(values, weights, synergy, capacity):
    '''Exact QKP solver. Checks every subset of items.'''
    n = len(values)
    best_x, best_value = tuple([0] * n), 0

    for bits in itertools.product([0, 1], repeat=n):
        if qkp_weight(weights, bits) <= capacity:
            val = qkp_value(values, synergy, bits)
            if val > best_value:
                best_value = val
                best_x = bits

    return best_x, best_value


bf_x, bf_value = qkp_brute_force(example_values, example_weights, example_synergy, example_capacity)
print("Brute force solution: ", bf_x)
print("Brute force value:    ", bf_value)
print("Weight used:           ", qkp_weight(example_weights, bf_x), "/", example_capacity)

### 1.3.2 A fast classical fallback: value-density greedy

A standard greedy for the plain knapsack sorts items by value-to-weight ratio and packs them in that order while capacity allows. We use exactly that rule here too. Notice what it deliberately ignores: it never looks at $q_{ij}$ at all, it only ever sees $v_j / w_j$.

In [ ]:
def qkp_greedy(values, weights, synergy, capacity):
    '''Greedy QKP heuristic. Packs items in decreasing value/weight order,
    completely ignoring the pairwise synergy terms.'''
    n = len(values)
    order = sorted(range(n), key=lambda j: values[j] / weights[j], reverse=True)

    x = [0] * n
    remaining = capacity
    for j in order:
        if weights[j] <= remaining:
            x[j] = 1
            remaining -= weights[j]

    return tuple(x), qkp_value(values, synergy, x)


greedy_x, greedy_value = qkp_greedy(example_values, example_weights, example_synergy, example_capacity)
print("Greedy solution: ", greedy_x)
print("Greedy value:    ", greedy_value)
print("Weight used:      ", qkp_weight(example_weights, greedy_x), "/", example_capacity)
print()
print("Gap versus brute force optimum:", bf_value - greedy_value)

**Practice task 1.** Run the two cells above as they are, then look at the two solutions returned. The greedy rule packs items purely by $v_j / w_j$, so it has no way to notice that items 1 and 4 together are worth an extra $q_{14}=10$. Write one sentence explaining, in terms of what information the greedy rule has access to, why it cannot be fixed just by sorting on a different single number per item.

## 1.4 Formulating QKP as a QUBO

This follows the same three steps as Lecture 7: turn the inequality into an equality with a binary-expanded slack, build the penalty term, then read off the QUBO matrix from the expanded objective. The only change from plain knapsack is that the QKP objective already has quadratic terms in it, they simply become extra off-diagonal entries in $Q$ alongside the ones the penalty introduces.

**Step 1, slack variable.** With $S_{\max} = W$, write
$$\sum_j w_j x_j + \sum_{k=0}^{K-1} 2^k y_k = W, \qquad K = \lceil \log_2(W+1) \rceil.$$

**Step 2, penalty term.** With coefficient $\lambda$,
$$P = \lambda\Big(\sum_j w_j x_j + \sum_k 2^k y_k - W\Big)^2.$$

**Step 3, full QUBO objective (maximization form).**
$$F(x, y) = \underbrace{\sum_j v_j x_j + \sum_{i<j} q_{ij} x_i x_j}_{\text{QKP objective}} - \lambda\Big(\sum_j w_j x_j + \sum_k 2^k y_k - W\Big)^2$$

We minimize $-F$, so the diagonal and off-diagonal entries of $Q$ collect: the linear values (negated), the pairwise synergies (negated), and the expansion of the penalty square exactly as derived in Lecture 7 for plain knapsack. As before, a comfortable rule of thumb is $\lambda \ge 1 + \max_j v_j$, adjusted upward a little to also outweigh the synergy terms.

In [ ]:
def qkp_qubo(values, weights, synergy, capacity, penalty=None):
    '''Builds the QUBO dictionary Q for a Quadratic Knapsack instance.
    Variables 0..n-1 are the items, variables n..n+K-1 are the binary
    expansion of the slack. Q is returned upper-triangular.'''
    n = len(values)
    K = int(np.ceil(np.log2(capacity + 1))) if capacity > 0 else 1

    if penalty is None:
        synergy_budget = sum(abs(q) for q in synergy.values())
        penalty = 1 + max(values) + synergy_budget

    Q = {}
    def add(a, b, amount):
        key = (a, b) if a <= b else (b, a)
        Q[key] = Q.get(key, 0) + amount

    # QKP objective, negated because we minimize -F
    for j in range(n):
        add(j, j, -values[j])
    for (i, j), q in synergy.items():
        add(i, j, -q)

    # penalty: lambda * (sum_a c_a z_a - W)^2, z = (x_0..x_{n-1}, y_0..y_{K-1})
    coeffs = list(weights) + [2 ** k for k in range(K)]
    m = n + K
    for a in range(m):
        add(a, a, penalty * (coeffs[a] ** 2 - 2 * capacity * coeffs[a]))
        for b in range(a + 1, m):
            add(a, b, 2 * penalty * coeffs[a] * coeffs[b])

    return Q, n, K, penalty


Q_example, n_items, K_slack, lam_used = qkp_qubo(example_values, example_weights, example_synergy, example_capacity)
print(f"n = {n_items} item variables, K = {K_slack} slack variables, total = {n_items + K_slack} binary variables")
print(f"penalty lambda used: {lam_used}")
print(f"Number of nonzero entries in Q: {len(Q_example)}")

### 1.4.1 Looking at the $Q$ matrix

Turning the dictionary into a dense matrix and plotting it makes the structure from Lecture 7 visible directly: the matrix is dense, and every pair of variables interacts once the capacity penalty is expanded.

In [ ]:
def qubo_dict_to_matrix(Q, m):
    M = np.zeros((m, m))
    for (a, b), val in Q.items():
        M[a, b] += val
        if a != b:
            M[b, a] += val  # mirror for a symmetric picture only, Q itself stays upper-triangular
    return M


m_total = n_items + K_slack
Q_matrix = qubo_dict_to_matrix(Q_example, m_total)

labels = [f"x{j+1}" for j in range(n_items)] + [f"y{k}" for k in range(K_slack)]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(Q_matrix, cmap="RdBu_r", vmin=-np.max(np.abs(Q_matrix)), vmax=np.max(np.abs(Q_matrix)))
ax.set_xticks(range(m_total))
ax.set_yticks(range(m_total))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_title("QKP QUBO matrix Q (symmetric view)")
plt.colorbar(im, ax=ax, label="coefficient")
plt.tight_layout()
plt.show()

np.set_printoptions(precision=1, suppress=True)
print(Q_matrix)

**Practice task 2.** Look at the block of the matrix corresponding to $x_1 \ldots x_4$ (top-left $4\times4$ block). Two forces both write into this block: the penalty term (which makes every pair of items interact, as in Lecture 7) and the synergy terms $q_{ij}$ themselves. Pick the entry for $(x_1, x_4)$ and write down, by hand, the two separate contributions that add up to it: $-q_{14}$ from the objective and $2\lambda w_1 w_4$ from the penalty.

## 1.5 Solving the QUBO with simulated annealing

In [ ]:
bqm_example = dimod.BinaryQuadraticModel.from_qubo(Q_example)
result_example = sampler.sample(bqm_example, num_reads=500, num_sweeps=2000, seed=RNG_SEED)

best_sample = result_example.first.sample
qubo_x = tuple(int(best_sample[j]) for j in range(n_items))
qubo_weight = qkp_weight(example_weights, qubo_x)
qubo_value = qkp_value(example_values, example_synergy, qubo_x)
qubo_feasible = qubo_weight <= example_capacity

print("QUBO / simulated annealing solution:", qubo_x)
print("Weight used:  ", qubo_weight, "/", example_capacity)
print("Value:        ", qubo_value)
print("Feasible:     ", qubo_feasible)

## 1.6 Comparing all three methods on the worked example

In [ ]:
print(f"{'Method':<28}{'Solution':<16}{'Value':<10}{'Feasible'}")
print("-" * 62)
print(f"{'Brute force (optimal)':<28}{str(bf_x):<16}{bf_value:<10}{qkp_weight(example_weights, bf_x) <= example_capacity}")
print(f"{'Greedy (value/weight)':<28}{str(greedy_x):<16}{greedy_value:<10}{qkp_weight(example_weights, greedy_x) <= example_capacity}")
print(f"{'QUBO + simulated annealing':<28}{str(qubo_x):<16}{qubo_value:<10}{qubo_feasible}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
methods = ["Brute force\n(optimal)", "Greedy\n(value/weight)", "QUBO +\nsimulated annealing"]
values_plot = [bf_value, greedy_value, qubo_value]
colors = ["#1f4e79", "#e0a458", "#4c8c4a"]

bars = ax.bar(methods, values_plot, color=colors)
ax.set_ylabel("Objective value (linear + synergy)")
ax.set_title("QKP worked example: value found by each method")
ax.axhline(bf_value, color="gray", linestyle="--", linewidth=1, label="optimal value")
for bar, val in zip(bars, values_plot):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, str(val), ha="center")
ax.legend()
plt.tight_layout()
plt.show()

**Practice task 3.** Run the cell above, then go back to section 1.5 and change `num_reads=500` to `num_reads=20`, re-run sections 1.5 and 1.6. Does simulated annealing still find the optimum every time you re-run it, or does it start to miss occasionally? This is the same lesson as Lab 2: a metaheuristic trades a runtime guarantee for a correctness guarantee, and how much you can trade away depends on how much search effort you give it.

## 1.7 Where classical brute force struggles

We now grow a random QKP instance and time brute force against greedy, the same style of experiment as the earlier labs.

In [ ]:
def random_qkp_instance(n, capacity_fraction=0.5, seed=0):
    rgen = np.random.default_rng(seed)
    values = rgen.integers(2, 15, size=n).tolist()
    weights = rgen.integers(2, 15, size=n).tolist()
    capacity = int(capacity_fraction * sum(weights))

    synergy = {}
    for i, j in itertools.combinations(range(n), 2):
        if rgen.random() < 0.5:
            synergy[(i, j)] = int(rgen.integers(-6, 9))

    return values, weights, synergy, capacity


node_range = range(4, 21, 2)
qkp_times = []

for n in node_range:
    v, w, s, cap = random_qkp_instance(n, seed=RNG_SEED)
    start = time.time()
    qkp_brute_force(v, w, s, cap)
    elapsed = time.time() - start
    qkp_times.append(elapsed)
    print(f"n = {n:2d} items   time = {elapsed:8.4f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(node_range), qkp_times, marker="o", color="#1f4e79")
ax.set_yscale("log")
ax.set_xlabel("Number of items (n)")
ax.set_ylabel("Runtime in seconds (log scale)")
ax.set_title("Brute force QKP: runtime grows exponentially with n")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

**Practice task 4.** Run the cell above, then change `node_range` to `range(4, 25, 2)` and re-run both cells. Notice that QKP brute force is already slower at the same $n$ than plain 0-1 knapsack brute force would be, since checking feasibility is identical but computing the value now costs $O(n^2)$ per subset instead of $O(n)$, on top of still checking $2^n$ subsets.

In [ ]:
n_gap = 14
v_gap, w_gap, s_gap, cap_gap = random_qkp_instance(n_gap, capacity_fraction=0.45, seed=RNG_SEED + 4)

start = time.time()
bf_x_gap, bf_value_gap = qkp_brute_force(v_gap, w_gap, s_gap, cap_gap)
bf_time_gap = time.time() - start

start = time.time()
greedy_x_gap, greedy_value_gap = qkp_greedy(v_gap, w_gap, s_gap, cap_gap)
greedy_time_gap = time.time() - start

Q_gap, n_gap_vars, K_gap, lam_gap = qkp_qubo(v_gap, w_gap, s_gap, cap_gap)
bqm_gap = dimod.BinaryQuadraticModel.from_qubo(Q_gap)
start = time.time()
result_gap = sampler.sample(bqm_gap, num_reads=500, num_sweeps=2000, seed=RNG_SEED)
sa_time_gap = time.time() - start
qubo_x_gap = tuple(int(result_gap.first.sample[j]) for j in range(n_gap_vars))
qubo_weight_gap = qkp_weight(w_gap, qubo_x_gap)
qubo_value_gap = qkp_value(v_gap, s_gap, qubo_x_gap)
qubo_feasible_gap = qubo_weight_gap <= cap_gap

print(f"n = {n_gap} items, capacity = {cap_gap}")
print()
print(f"{'Method':<28}{'Value':<10}{'Feasible':<12}{'Time (s)'}")
print("-" * 60)
print(f"{'Brute force (optimal)':<28}{bf_value_gap:<10}{True:<12}{bf_time_gap:.4f}")
print(f"{'Greedy (value/weight)':<28}{greedy_value_gap:<10}{True:<12}{greedy_time_gap:.6f}")
print(f"{'QUBO + simulated annealing':<28}{qubo_value_gap:<10}{str(qubo_feasible_gap):<12}{sa_time_gap:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
methods = ["Brute force\n(optimal)", "Greedy", "QUBO + SA"]
vals = [bf_value_gap, greedy_value_gap, qubo_value_gap]
times_s = [bf_time_gap, greedy_time_gap, sa_time_gap]

ax2 = ax.twinx()
bars = ax.bar([m for m in methods], vals, color="#1f4e79", alpha=0.75, label="value")
ax2.plot(methods, times_s, color="#d62728", marker="o", linewidth=2, label="runtime (s)")
ax2.set_yscale("log")

ax.set_ylabel("Objective value")
ax2.set_ylabel("Runtime in seconds (log scale)")
ax.set_title(f"n = {n_gap} item QKP instance: value and runtime side by side")
fig.tight_layout()
plt.show()

**Practice task 5.** Run the cell above, then change `n_gap = 14` to `n_gap = 18` and re-run this subsection. At what point does brute force become slow enough that you would not want to wait for it in a live demo? Report the value gap between greedy and the optimum at this larger size.

> ### Explore further: Quadratic Knapsack Problem
>
> - **[Pisinger, "The quadratic knapsack problem, a survey," Discrete Applied Mathematics, 2007](https://doi.org/10.1016/j.dam.2006.08.007)**
>   The standard survey of exact algorithms, upper bounds, and heuristics for QKP, including the reformulation and Lagrangian techniques the field has built up over decades.
>
> - **[Glover, Kochenberger, Du, "Quantum Bridge Analytics I: a tutorial on formulating and using QUBO models," Annals of Operations Research, 2022](https://link.springer.com/article/10.1007/s10479-022-04634-2)**
>   The QKP numerical example used in section 1.2 is taken directly from this tutorial's worked example, which walks through the same QUBO derivation with a slightly different penalty value.
>
> - **[D-Wave Ocean documentation: dimod](https://docs.ocean.dwavesys.com/en/stable/docs_dimod/)** and **[Ocean documentation: dwave-neal](https://docs.ocean.dwavesys.com/en/stable/docs_neal/)**
>   Reference documentation for the `BinaryQuadraticModel` and `SimulatedAnnealingSampler` classes used throughout this notebook.

## 1.8 QKP summary

| Method | Guarantees optimum | Scales past n ~ 20 | Sees pairwise synergy |
|---|---|---|---|
| Brute force | Yes | No | Yes, but pays $O(n^2)$ per subset on top of $2^n$ subsets |
| Value/weight greedy | No | Yes | No, sorts on a per-item ratio only |
| Simulated annealing (QUBO) | No, but close when penalty and search effort are adequate | Yes | Yes, synergy terms sit directly in $Q$ |

The lesson to carry into Part 2: once an objective stops being a simple sum over individual items, a greedy rule built for the simple sum stops being a safe default, and you need either an exact search or a solver that actually sees the full objective, like the QUBO does here.

---
# Part 2: Polynomial Unconstrained Binary Optimization (PUBO)

QKP added pairwise terms $q_{ij}x_ix_j$ to a linear objective. Nothing stops the same idea going further: sometimes the value of choosing three items together, or four, differs from the sum of their pairs. This gives a **Polynomial Unconstrained Binary Optimization** problem:

$$\max \sum_j v_j x_j + \sum_{i<j} q_{ij} x_i x_j + \sum_{i<j<k} c_{ijk}\, x_i x_j x_k + \cdots$$

QUBO restricts the degree of this polynomial to at most 2. PUBO allows any degree. Quantum annealers and most QUBO solvers only accept degree $\le 2$ inputs, so a PUBO with higher-degree terms has to be **quadratized**, rewritten as an equivalent QUBO in a larger number of variables, before it can be handed to a solver like `neal`.

## 2.1 Where a higher-order term comes from

Consider a bonus that only appears when three specific items, $i$, $j$, and $k$, are all selected together:

$$c_{ijk}\, x_i x_j x_k = \begin{cases} c_{ijk}, & x_i = x_j = x_k = 1 \\ 0, & \text{otherwise} \end{cases}$$

This term cannot be written as any combination of linear and pairwise terms. It genuinely needs degree 3, and the objective containing it is a PUBO, not a QUBO.

**Applications where this arises directly:**
- **Chemical or drug synergy.** A three-drug combination therapy can have an effect that is not predictable from any pair of the three drugs alone.
- **Multi-way circuit or hardware co-design.** Three components placed together on a board can create an interference pattern that no pairwise interaction term captures.
- **Team formation.** A three-person subteam can be unusually effective (or unusually dysfunctional) together in a way that pairwise compatibility scores between two people at a time do not predict.

## 2.2 A worked PUBO example

Five binary decisions $x_1, \ldots, x_5$. Individual values, two pairwise synergies, and two three-way synergies:

| Term | Coefficient | Meaning |
|---|---|---|
| $v_1, \ldots, v_5$ | $3, 4, 2, 5, 3$ | individual value of each choice |
| $q_{12}$ | $+6$ | items 1 and 2 reinforce each other |
| $q_{25}$ | $-3$ | items 2 and 5 are partly redundant |
| $c_{123}$ | $+8$ | items 1, 2, 3 together unlock a bonus |
| $c_{345}$ | $-5$ | items 3, 4, 5 together create a penalty |

$$\max\ 3x_1+4x_2+2x_3+5x_4+3x_5 + 6x_1x_2 - 3x_2x_5 + 8x_1x_2x_3 - 5x_3x_4x_5$$

There is no capacity constraint here, this is the "unconstrained" in PUBO: every one of the $2^5 = 32$ combinations is feasible, we are purely looking for the one with the highest objective value.

In [ ]:
# PUBO terms stored as {tuple_of_variable_indices: coefficient}, 0-indexed.
# A 1-tuple is a linear term, a 2-tuple a pairwise term, a 3-tuple a three-way term.
pubo_terms = {
    (0,): 3, (1,): 4, (2,): 2, (3,): 5, (4,): 3,
    (0, 1): 6,
    (1, 4): -3,
    (0, 1, 2): 8,
    (2, 3, 4): -5,
}
n_pubo = 5

def pubo_value(terms, x):
    total = 0
    for idxs, coeff in terms.items():
        term = coeff
        for i in idxs:
            term *= x[i]
        total += term
    return total

# sanity check: all items selected
x_all = (1, 1, 1, 1, 1)
print("Value with every item selected:", pubo_value(pubo_terms, x_all))
print("(3+4+2+5+3) + 6 + (-3) + 8 + (-5) =", (3+4+2+5+3) + 6 - 3 + 8 - 5)

## 2.3 Classical approaches

Same two-method comparison as before: brute force over every binary string, and a greedy local search that cannot see terms it has not already committed to.

### 2.3.1 Brute force

For $n$ binary variables there are $2^n$ strings to check, regardless of the degree of the polynomial. Evaluating a degree-3 objective on each string costs a little more per string than a linear one, but the search space itself is exactly as large as it would be for plain QUBO.

In [ ]:
def pubo_brute_force(terms, n):
    '''Exact PUBO solver. Evaluates every binary string.'''
    best_x, best_value = tuple([0] * n), pubo_value(terms, [0] * n)

    for bits in itertools.product([0, 1], repeat=n):
        val = pubo_value(terms, bits)
        if val > best_value:
            best_value = val
            best_x = bits

    return best_x, best_value


pubo_bf_x, pubo_bf_value = pubo_brute_force(pubo_terms, n_pubo)
print("Brute force solution:", pubo_bf_x)
print("Brute force value:   ", pubo_bf_value)

### 2.3.2 A fast classical fallback: greedy bit-flip local search

Start from an arbitrary binary string, and repeatedly flip whichever single bit improves the objective by the most. Stop when no single flip helps any more. This is fast, always terminates, and always finds *some* local optimum, but nothing stops it from getting stuck one flip away from the true optimum if the three-way terms only pay off after a combination of flips that individually looks unhelpful.

In [ ]:
def pubo_greedy_local_search(terms, n, start=None):
    '''Steepest-ascent bit-flip local search for PUBO.
    Repeatedly flips the single bit that improves the objective the most,
    stops at the first local optimum.'''
    x = list(start) if start is not None else [0] * n
    current_value = pubo_value(terms, x)

    improved = True
    while improved:
        improved = False
        best_gain = 0
        best_flip = None
        for i in range(n):
            x[i] ^= 1
            new_value = pubo_value(terms, x)
            gain = new_value - current_value
            x[i] ^= 1
            if gain > best_gain:
                best_gain = gain
                best_flip = i
        if best_flip is not None:
            x[best_flip] ^= 1
            current_value += best_gain
            improved = True

    return tuple(x), current_value


greedy_pubo_x, greedy_pubo_value = pubo_greedy_local_search(pubo_terms, n_pubo, start=[0, 0, 0, 0, 0])
print("Greedy local search solution (starting from all zeros):", greedy_pubo_x)
print("Greedy local search value:                              ", greedy_pubo_value)
print()
print("Gap versus brute force optimum:", pubo_bf_value - greedy_pubo_value)

**Practice task 6.** Run the cell above, then call `pubo_greedy_local_search(pubo_terms, n_pubo, start=[1, 1, 1, 1, 1])` with the opposite starting point and compare. Local search only ever looks one flip ahead, so different starting points can land in different local optima. Report both results and say which start reaches the true optimum from section 2.3.1.

## 2.4 Quadratization: reducing a cubic term to a QUBO

`neal` and most QUBO solvers only accept degree 2. Every cubic term needs to be replaced before we can build $Q$. This uses the same Rosenberg substitution from Lecture 7.

**Algorithm, for a cubic term $c\, x_i x_j x_k$.**

1. Introduce one new binary variable $w_{ij} = x_i x_j$ (an ancilla, "gadget").
2. Replace $x_i x_j x_k$ by $w_{ij} x_k$, which is now degree 2.
3. Add a penalty that forces $w_{ij}$ to actually equal $x_i x_j$:
$$\Delta(x_i, x_j, w_{ij}) = 3w_{ij} + x_i x_j - 2x_i w_{ij} - 2x_j w_{ij}$$

This penalty is zero exactly when $w_{ij}$ agrees with $x_i x_j$, and strictly positive at the two inconsistent assignments, so a large enough coefficient $\lambda_{\text{aux}}$ makes those assignments strictly worse in the objective, ruling them out.

In [ ]:
def rosenberg_delta(xi, xj, w):
    '''The four cases of the Rosenberg penalty, to verify it is zero exactly
    when w = xi and xj (i.e. w equals xi AND xj).'''
    return 3 * w + xi * xj - 2 * xi * w - 2 * xj * w


print(f"{'xi':<4}{'xj':<4}{'w=xi*xj':<10}{'w used':<8}{'delta'}")
for xi, xj in itertools.product([0, 1], repeat=2):
    correct_w = xi * xj
    for w in [0, 1]:
        print(f"{xi:<4}{xj:<4}{correct_w:<10}{w:<8}{rosenberg_delta(xi, xj, w)}")

**Practice task 7.** Look at the table above. Confirm by eye that `delta = 0` only ever happens when `w used` matches `w=xi*xj`, and that every mismatched row has `delta > 0`. This is exactly what makes the penalty enforceable: any solver minimizing $-F + \lambda_{\text{aux}}\Delta$ pays a strictly positive cost for choosing an inconsistent $w$, as long as $\lambda_{\text{aux}} > 0$.

### 2.4.1 Quadratizing the full example

Our objective has two cubic terms, $8x_1x_2x_3$ and $-5x_3x_4x_5$. Each needs its own ancilla. We introduce $w_{12} = x_1x_2$ for the first and $w_{34} = x_3x_4$ for the second, and add a Rosenberg penalty for each.

In [ ]:
def quadratize_pubo(terms, n, aux_penalty):
    '''Reduces a PUBO (terms of degree <= 3) to a QUBO dictionary.
    Returns Q (upper-triangular dict), the ancilla-variable index map, and
    the total variable count (original n plus one ancilla per cubic term).'''
    Q = {}
    def add(a, b, amount):
        key = (a, b) if a <= b else (b, a)
        Q[key] = Q.get(key, 0) + amount

    ancilla_of_pair = {}
    next_index = n

    # first pass: figure out which pairs need an ancilla (one per distinct
    # (i, j) pair that appears as the "base pair" of a cubic term)
    for idxs in terms:
        if len(idxs) == 3:
            i, j, k = idxs
            pair = (i, j)
            if pair not in ancilla_of_pair:
                ancilla_of_pair[pair] = next_index
                next_index += 1

    total_vars = next_index

    # second pass: build Q, negated because we minimize -F
    for idxs, coeff in terms.items():
        if len(idxs) == 1:
            (i,) = idxs
            add(i, i, -coeff)
        elif len(idxs) == 2:
            i, j = idxs
            add(i, j, -coeff)
        elif len(idxs) == 3:
            i, j, k = idxs
            w = ancilla_of_pair[(i, j)]
            # -coeff * w * x_k  (degree 2 now)
            add(w, k, -coeff)
            # Rosenberg penalty: lambda_aux * (3w + xi*xj - 2*xi*w - 2*xj*w)
            add(w, w, aux_penalty * 3)
            add(i, j, aux_penalty * 1)
            add(i, w, aux_penalty * -2)
            add(j, w, aux_penalty * -2)
        else:
            raise ValueError("only linear, pairwise, and cubic terms are supported")

    return Q, ancilla_of_pair, total_vars


aux_lambda = 20  # comfortably larger than the largest cubic coefficient (8), see section 1.4 rule of thumb
Q_pubo, ancilla_map, total_pubo_vars = quadratize_pubo(pubo_terms, n_pubo, aux_lambda)

print("Ancilla variables introduced:")
for (i, j), w in ancilla_map.items():
    print(f"  w_{i}{j} = x{i+1}*x{j+1}  ->  variable index {w}")
print(f"\nTotal QUBO variables: {total_pubo_vars} ({n_pubo} original + {len(ancilla_map)} ancilla)")

### 2.4.2 The $Q$ matrix after quadratization

In [ ]:
Q_pubo_matrix = qubo_dict_to_matrix(Q_pubo, total_pubo_vars)
pubo_labels = [f"x{j+1}" for j in range(n_pubo)] + [f"w{i+1}{j+1}" for (i, j) in ancilla_map]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(Q_pubo_matrix, cmap="RdBu_r",
               vmin=-np.max(np.abs(Q_pubo_matrix)), vmax=np.max(np.abs(Q_pubo_matrix)))
ax.set_xticks(range(total_pubo_vars))
ax.set_yticks(range(total_pubo_vars))
ax.set_xticklabels(pubo_labels)
ax.set_yticklabels(pubo_labels)
ax.set_title("Quadratized PUBO: QUBO matrix Q (symmetric view)")
plt.colorbar(im, ax=ax, label="coefficient")
plt.tight_layout()
plt.show()

np.set_printoptions(precision=1, suppress=True)
print(Q_pubo_matrix)

Notice the two ancilla rows and columns sitting outside the original $5\times5$ block: they exist purely to carry the Rosenberg penalty, and they do not correspond to any decision the person choosing $x_1,\ldots,x_5$ actually makes. This is the "one new binary variable per cubic interaction" cost mentioned in Lecture 7, made concrete.

## 2.5 Solving the quadratized QUBO with simulated annealing

In [ ]:
bqm_pubo = dimod.BinaryQuadraticModel.from_qubo(Q_pubo)
result_pubo = sampler.sample(bqm_pubo, num_reads=1000, num_sweeps=2000, seed=RNG_SEED)

best_pubo_sample = result_pubo.first.sample
qubo_pubo_x = tuple(int(best_pubo_sample[i]) for i in range(n_pubo))
qubo_pubo_value = pubo_value(pubo_terms, qubo_pubo_x)

# check that every ancilla actually equals the product it was supposed to represent
ancilla_consistent = all(
    best_pubo_sample[w] == best_pubo_sample[i] * best_pubo_sample[j]
    for (i, j), w in ancilla_map.items()
)

print("QUBO / simulated annealing solution (original variables):", qubo_pubo_x)
print("Objective value (evaluated on the original PUBO):        ", qubo_pubo_value)
print("All ancilla variables consistent with xi*xj:             ", ancilla_consistent)

**Practice task 8.** Run the cell above, then lower `aux_lambda` in section 2.4.1 from `20` down to `1` and re-run sections 2.4.1 through 2.5. Does `ancilla_consistent` stay `True`? This is the same lesson as the penalty coefficient exercise in Part 1: the Rosenberg penalty is only enforced if it is large enough to make an inconsistent ancilla strictly worse than a consistent one, and a value close to (or smaller than) the cubic coefficients it needs to dominate is not large enough.

## 2.6 Comparing all three methods on the worked PUBO example

In [ ]:
print(f"{'Method':<32}{'Solution':<18}{'Value'}")
print("-" * 62)
print(f"{'Brute force (optimal)':<32}{str(pubo_bf_x):<18}{pubo_bf_value}")
print(f"{'Greedy local search':<32}{str(greedy_pubo_x):<18}{greedy_pubo_value}")
print(f"{'Quadratized QUBO + sim. annealing':<32}{str(qubo_pubo_x):<18}{qubo_pubo_value}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
methods = ["Brute force\n(optimal)", "Greedy\nlocal search", "Quadratized QUBO\n+ sim. annealing"]
values_plot = [pubo_bf_value, greedy_pubo_value, qubo_pubo_value]
colors = ["#1f4e79", "#e0a458", "#4c8c4a"]

bars = ax.bar(methods, values_plot, color=colors)
ax.axhline(pubo_bf_value, color="gray", linestyle="--", linewidth=1, label="optimal value")
ax.set_ylabel("Objective value")
ax.set_title("PUBO worked example: value found by each method")
for bar, val in zip(bars, values_plot):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.2, str(val), ha="center")
ax.legend()
plt.tight_layout()
plt.show()

## 2.7 Where brute force struggles, and how variable count grows with degree

The search space for PUBO brute force is $2^n$, exactly like QUBO, since the degree of the polynomial changes how expensive each evaluation is but not how many binary strings exist. What actually explodes with higher-order terms is the **size of the quadratized QUBO**: every cubic term can need its own ancilla variable.

In [ ]:
pubo_node_range = range(4, 21, 2)
pubo_times = []

for n in pubo_node_range:
    rgen = np.random.default_rng(RNG_SEED + n)
    terms = {(i,): int(rgen.integers(1, 6)) for i in range(n)}
    for i, j in itertools.combinations(range(n), 2):
        if rgen.random() < 0.15:
            terms[(i, j)] = int(rgen.integers(-5, 6))
    for i, j, k in itertools.combinations(range(n), 3):
        if rgen.random() < 0.03:
            terms[(i, j, k)] = int(rgen.integers(-6, 7))

    start = time.time()
    pubo_brute_force(terms, n)
    elapsed = time.time() - start
    pubo_times.append(elapsed)
    print(f"n = {n:2d} variables, {sum(1 for k in terms if len(k) == 3)} cubic terms   time = {elapsed:8.4f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(pubo_node_range), pubo_times, marker="o", color="#1f4e79")
ax.set_yscale("log")
ax.set_xlabel("Number of variables (n)")
ax.set_ylabel("Runtime in seconds (log scale)")
ax.set_title("Brute force PUBO: runtime grows exponentially with n, same as QUBO")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# variable overhead from quadratization as the number of cubic terms grows
cubic_term_counts = list(range(0, 11))
overhead_vars = [n_pubo + count for count in cubic_term_counts]  # one ancilla per distinct base pair, worst case one per cubic term

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(cubic_term_counts, overhead_vars, marker="o", color="#8c564b")
ax.set_xlabel("Number of cubic terms introduced")
ax.set_ylabel("Total QUBO variables after quadratization")
ax.set_title(f"Variable overhead of Rosenberg quadratization, starting from n={n_pubo}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Practice task 9.** Run the two timing cells above, then change `pubo_node_range` to `range(4, 25, 2)` and re-run. The brute force curve should look almost identical in shape to the QKP curve from Part 1 and the earlier MVC and graph coloring labs, since it is driven purely by $2^n$, not by the degree of the polynomial being evaluated.

> ### Explore further: PUBO and Quadratization
>
> - **[Rosenberg, "Reduction of bivalent maximization to the quadratic case," Cahiers du CERO, 1975](https://scholar.google.com/scholar?q=Rosenberg+1975+Reduction+of+bivalent+maximization+to+the+quadratic+case)**
>   The original reduction used in section 2.4, showing that any pseudo-Boolean maximization can be reduced to a quadratic one by adding auxiliary variables.
>
> - **[Boros and Hammer, "Pseudo-Boolean optimization," Discrete Applied Mathematics, 2002](https://doi.org/10.1016/S0166-218X(01)00341-9)**
>   A broader survey of pseudo-Boolean (PUBO) optimization, including several quadratization techniques beyond Rosenberg's, and their relative variable overhead.
>
> - **[D-Wave Ocean documentation: dimod higher order composites](https://docs.ocean.dwavesys.com/en/stable/docs_dimod/reference/higherorder.html)**
>   Ocean's own tools for building and reducing higher-order (PUBO-style) binary polynomials into `BinaryQuadraticModel` objects, a ready-made alternative to hand-rolling the Rosenberg substitution as we did above.

## 2.8 PUBO summary

| Method | Guarantees optimum | Handles degree > 2 directly | Extra cost |
|---|---|---|---|
| Brute force | Yes | Yes | $O(2^n)$ regardless of degree, cost per evaluation grows slowly with degree |
| Greedy local search | No | Yes | Fast, but stuck at whatever local optimum it reaches from its starting point |
| Quadratized QUBO + simulated annealing | No, but close with a large enough auxiliary penalty | No, needs one ancilla per cubic term first | Extra variables per cubic term, extra penalty coefficient to tune correctly |

Carrying both parts of this lab forward: whenever an objective has interactions beyond single items, a solver (or a person) that only sees individual items in isolation cannot reconstruct the true optimum from local information alone. QUBO extends that visibility to pairs, and quadratization is the price of extending it further to triples and beyond while still using QUBO hardware and solvers.

---
# Part 3: Exercises

Complete these after the live session, on your own. None of them require writing a new algorithm from scratch, only changing a parameter, an instance, or a line or two in cells you already ran. Re-run the relevant cells after each change and record what you observe.

**Exercise 1 (QKP, easy).** In section 1.7, change `capacity_fraction=0.5` to `capacity_fraction=0.25` inside `random_qkp_instance`, and re-run the timing loop. Does brute force reach the same `n` values in a similar amount of time, or does a tighter capacity change the runtime? Explain in one or two sentences why the capacity fraction does, or does not, affect how many subsets brute force has to check.

**Exercise 2 (QKP, medium).** In section 1.5, try three different values of `penalty` when building `Q_example` (for example `5`, `40`, and `200`), by calling `qkp_qubo(example_values, example_weights, example_synergy, example_capacity, penalty=5)` directly instead of letting the function pick one automatically. For each value, report the QUBO solution, its value, and whether it is feasible. At what point does increasing the penalty further stop changing the result, and at what point does it start to hurt the solution quality?

**Exercise 3 (PUBO, easy to medium).** Add a new three-way synergy term to `pubo_terms` from section 2.2, for example `(1, 3, 4): 7`, and re-run sections 2.3 through 2.6. Report the new brute force optimum, the new number of ancilla variables introduced by quadratization, and whether simulated annealing still finds the optimal value.

**Exercise 4 (PUBO, medium to hard).** Write a short paragraph (4 to 6 sentences) comparing the two "extra cost" mechanisms you saw in this lab: the capacity penalty in QKP (Part 1) and the Rosenberg ancilla penalty in PUBO (Part 2). Both are penalty terms added to an unconstrained objective, but they are enforcing two different kinds of thing. What is each one actually enforcing, and what would go wrong in the returned solution if each one were set too small?

## What to submit

Save this notebook with all cells executed and their output and plots visible, along with your answers to the four exercises above (a few sentences each, plus the specific numbers you observed where asked). If you explored beyond what the exercises ask, leave a short note describing what you changed and what effect it had.